<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will prioritize pages using a simple directional score based on their March search-performance signals. Pages with stronger search exposure and weaker search performance receive higher priority for review.

The baseline uses **March impressions, CTR, and average position**. The score is intended for **prioritization and decision-support**, not as proof that a page needs a specific content change.

### Reason codes

* **HIGH_EXPOSURE** — the page has relatively high March impressions, so changes could affect a meaningful amount of search exposure.
* **LOW_CTR_OPPORTUNITY** — the page has relatively low CTR compared with other pages, making it worth reviewing, but this does not prove that a CTR fix will improve clicks.
* **WEAK_POSITION** — the page has a relatively poor March average search position, indicating weaker search visibility.
* **MULTI_SIGNAL** — the page shows more than one of the above conditions and therefore receives stronger review priority.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I convert the three confirmed March signals into percentile-based directional scores. Higher impressions increase priority because the page has more search exposure. Lower CTR increases priority because it represents a possible click opportunity. Worse average position increases priority because it represents weaker search visibility.

The three components are combined with equal weight. This is a baseline prioritization rule, not a predictive model. The score is used to decide which pages should be reviewed first.

In [3]:
# ML-07 — Section 2: Build the ranked queue

import duckdb
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Load Hugging Face access token
# ---------------------------------------------------------

from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

if not HF_TOKEN:
    raise ValueError("HF_Token was not found in Colab Secrets.")

# ---------------------------------------------------------
# 2. Connect to the FlyRank warehouse
# ---------------------------------------------------------

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

# ---------------------------------------------------------
# 3. Build the March feature frame
# ---------------------------------------------------------
# Only March data is used for the baseline score.
# April outcome data is NOT used here.

feature_sql = f"""
WITH march AS (
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions_march,

    SUM(gsc_clicks) AS gsc_clicks_march,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_ctr_march,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_avg_position_march,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE NULL
        END
    ) AS ga4_sessions_march

FROM march

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(feature_sql).df()

print("Feature frame shape:", features.shape)
display(features.head())

TimeoutException: Requesting secret HF_Token timed out. Secrets can only be fetched when running from the Colab UI.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.